# JalDrishti · Map any region with NISAR

Get a flood map for **any area and date** from NASA–ISRO **NISAR** L-band radar, using the same pipeline as the JalDrishti app (https://nayanip2001-max.github.io/JalDrishti/).

* **Option A:** draw nothing, just type a box and a date. The notebook finds the NISAR pass and the earlier "before" passes itself.
* **Option B:** upload your own radar GeoTIFFs (HH + HV backscatter). Files are checked first; incompatible data stops with a clear message.

You need a free **NASA Earthdata** login (https://urs.earthdata.nasa.gov/users/new) for option A. Runtime: about 5–20 minutes depending on the area.

**Honest notes:** accuracy was measured in Assam, India (held-out IoU 0.79) and Thailand (0.84 after adaptation). A map for a new region is **not validated there**. "Flood" means water that is new compared with the earlier passes of the same orbit track.

## 1 · Install (about 1 minute)

In [ ]:
!git clone -q --depth 1 https://github.com/nayanip2001-max/JalDrishti.git
%cd JalDrishti
!pip -q install -r requirements-live.txt matplotlib pystac-client planetary-computer

## 2 · NASA Earthdata login (option A only)
Your password is stored only in this Colab session (`~/.netrc`).

In [ ]:
import earthaccess
earthaccess.login(strategy="interactive", persist=True)

## 3A · Map an area and date with NISAR
On the JalDrishti dashboard, open **Map a Region**, choose the area and date, press **Check NISAR passes**, and copy the line it gives you into the box below. Or type it yourself: `name | west south east north | YYYY-MM-DD`.

In [ ]:
#@title Region and date
SETTINGS = "my_region | 80.10 12.85 80.30 13.10 | 2026-11-04"  #@param {type:"string"}
NAME, BBOX, DATE = [x.strip() for x in SETTINGS.split("|")]
WEST, SOUTH, EAST, NORTH = map(float, BBOX.split())
print(f"Mapping {NAME}: box W {WEST} S {SOUTH} E {EAST} N {NORTH}, date {DATE}")
!python scripts/map_region.py --name {NAME} --bbox {WEST} {SOUTH} {EAST} {NORTH} --date {DATE}

## 3B · …or upload your own GeoTIFFs
One post-event file and at least two earlier files of the same area. Put the date in each file name, e.g. `after_2026-08-20.tif`, `before_2026-07-27.tif`.

In [ ]:
from google.colab import files
up = files.upload()
names = sorted(up)
post = [n for n in names if n.lower().startswith(("after", "post"))]
refs = [n for n in names if n not in post]
assert post and len(refs) >= 2, "upload 1 'after_…' file and at least 2 earlier files"
NAME = "my_upload"
!python scripts/map_region.py --name {NAME} --upload {post[0]} --refs {' '.join(refs)}

## 4 · Result

In [ ]:
import json, glob
from IPython.display import Image, display
out = f"outputs/regions/{NAME}"
s = json.load(open(f"{out}/summary.json"))
print(json.dumps({k: s[k] for k in ("date", "track", "reference_dates", "areas_km2", "caveat")}, indent=1))
display(Image(glob.glob(f"{out}/quicklook_*.png")[0]))

In [ ]:
#@title Download the result (one zip: drop it on the dashboard's Map a Region page)
from google.colab import files
files.download(glob.glob(f"{out}/jaldrishti_*.zip")[0])